In [1]:
!python evaluations/text_cls_baseline/run.py --tasks sentiment relation corpus --epochs 10 --batch_size 16 --eval_dir evaluations/v2 --output_dir results/text_cls_baseline 2>&1

^C



Device: cuda
Eval dir: evaluations\v2
Output dir: results\text_cls_baseline
Model: microsoft/deberta-v3-base

Loading microsoft/deberta-v3-base for zero-shot embeddings…

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 33607.39it/s]

  Task: SENTIMENT  |  2208 samples  |  4 classes: ['negative', 'neutral', 'positive', 'nan']
  Label distribution: {'negative': 121, 'neutral': 181, 'positive': 115, 'nan': 1791}
  Chance baseline: 0.2500  (random)

[TF-IDF+LR] sentiment
  5-fold CV  Acc=1.0000±0.0000   F1-macro=1.0000±0.0000

[Zero-shot cosine] sentiment
  Top-1  Acc=0.0548  F1-macro=0.0590

  Classification Report:
              precision    recall  f1-score   support

    negative       0.06      0.90      0.11       121
     neutral       1.00      0.07      0.12       181
    positive       0.00      0.00      0.00       115
         nan       0.00      0.00      0.00      1791

    accuracy                           0.05      2208
   macro avg       0.26      0.24      0.06 

## Text Classification v2 — Corrected Baselines

Key improvements over the `run.py` above:

| Issue in `run.py` | Fix in `run_v2.py` |
|---|---|
| Data loaded from evaluation pickles (multi-subject copies → data leak → 100% TF-IDF) | Loaded directly from ZuCo DataFrame, deduplicated by `text uid` |
| 5-fold CV on same data pool as test | Proper `train` / `val` / `test` phase splits |
| SGD lr=1e-4 → NaN loss throughout training | AdamW lr=2e-5 + linear warmup schedule |
| No class weights → ignores imbalance | Balanced class-weighted cross-entropy |
| Crashes on relation (0-support classes) | `labels=list(range(n))` + `zero_division=0` in all reports |
| No benchmarking evaluation | X-cohort held-out subjects evaluated (inference only) |
| No T5 comparison | Flan-T5-Large encoder + linear head added |

**New metric**: F1-macro over **active classes only** (classes present in test set) is the headline number, so zero-recall on foundation/political_affiliation doesn't distort the relation score.

In [2]:
# Run v2 — sentiment, relation, corpus on main test set + benchmarking X-cohort
# Adjust --batch_size / --grad_accum_steps if GPU OOM with T5 (e.g. --batch_size 8 --grad_accum_steps 2)
# Add --no_t5 to skip the Flan-T5-Large encoder and save ~6GB VRAM

!python ../text_cls_baseline/run_v2.py \
    --main_df_path  data/tmp/zuco_eeg_label_8variants.df \
    --bench_df_path data/tmp/zuco_eeg_label_benchmarking_compat.df \
    --output_dir    results/text_cls_v2 \
    --tasks sentiment relation corpus \
    --epochs 15 \
    --batch_size 16 \
    --seed 42 \
    2>&1

python ../text_cls_baseline/run_v2.py --main_df_path  data/tmp/zuco_eeg_label_8variants.df --bench_df_path data/tmp/zuco_eeg_label_benchmarking_compat.df --output_dir    results/text_cls_v2 --tasks sentiment relation corpus --epochs 15 --batch_size 16 --seed 42 2>&1

^C



Device:    cuda
Main DF:   data/tmp/zuco_eeg_label_8variants.df
Bench DF:  data/tmp/zuco_eeg_label_benchmarking_compat.df
Output:    results\text_cls_v2

Loading zero-shot model: microsoft/deberta-v3-base …

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 32644.35it/s]

  Task: SENTIMENT  |  3 classes: ['negative', 'neutral', 'positive']

  Splits (deduplicated unique sentences):
    Train  n= 320  dist={'neutral': 104, 'negative': 103, 'positive': 113}
    Val    n=  40  dist={'positive': 16, 'neutral': 16, 'negative': 8}
    Test   n=  40  dist={'negative': 12, 'neutral': 17, 'positive': 11}

[TF-IDF + LR]  sentiment
  [TF-IDF+LR] val:  acc=0.4750  f1_macro(active)=0.4465
  [TF-IDF+LR] test:  acc=0.5000  f1_macro(active)=0.4875

[Zero-shot DeBERTa cosine]  sentiment
  TEST  acc=0.3000  f1_macro=0.1867

[Fine-tuned DeBERTa] sentiment  |  train=320  val=40  test=40

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 21919.71it/s]
  Ep 01/15  loss=nan  tr_acc=0.3250  val_a

In [3]:
"""Display v2 results as a comparison table."""
import json, pathlib, pandas as pd

results_dir = pathlib.Path("results/text_cls_v2")
rows = []
for task in ["sentiment", "relation", "corpus"]:
    fpath = results_dir / f"{task}_results.json"
    if not fpath.exists():
        print(f"[missing] {fpath}")
        continue
    r = json.loads(fpath.read_text())
    chance = r.get("chance_baseline", float("nan"))
    n_test = r.get("n_test", "?")

    method_keys = [
        ("Chance baseline",       None,                    "test"),
        ("TF-IDF + LR",          "tfidf_lr_test",          "test"),
        ("Zero-shot DeBERTa",     "zero_shot_cosine",       "test"),
        ("Fine-tuned DeBERTa",    "finetuned_deberta",      "test"),
        ("Flan-T5-Large encoder", "finetuned_t5encoder",    "test"),
        ("TF-IDF + LR",          "benchmarking.tfidf_lr",          "bench"),
        ("Zero-shot DeBERTa",     "benchmarking.zero_shot_cosine",  "bench"),
        ("Fine-tuned DeBERTa",    "benchmarking.finetuned_deberta", "bench"),
        ("Flan-T5-Large encoder", "benchmarking.finetuned_t5encoder", "bench"),
    ]
    for method, dotkey, split in method_keys:
        if dotkey is None:
            rows.append({"task": task, "split": split, "n": n_test,
                         "method": method, "accuracy": chance,
                         "f1_macro": None, "f1_weighted": None, "chance": chance})
            continue
        # support dot-notation for nested benchmarking keys
        d = r
        for k in dotkey.split("."):
            d = d.get(k, {}) if isinstance(d, dict) else {}
        if not d or d.get("skipped"):
            continue
        n = r.get(f"n_{split}", r.get("benchmarking", {}).get("n_samples", "?"))
        rows.append({"task": task, "split": split, "n": n,
                     "method": method,
                     "accuracy":    d.get("accuracy"),
                     "f1_macro":    d.get("f1_macro"),
                     "f1_weighted": d.get("f1_weighted"),
                     "chance": chance})

df = pd.DataFrame(rows)
df = df.sort_values(["task", "split", "f1_macro"], ascending=[True, True, False])

for split_name, grp in df.groupby("split"):
    print(f"\n── {split_name.upper()} SET ──────────────────────────────────────────")
    pivot = grp.pivot_table(
        index=["task", "method"],
        columns="split",
        values=["accuracy", "f1_macro", "f1_weighted"],
        aggfunc="first",
    )
    print(grp[["task", "n", "method", "accuracy", "f1_macro", "f1_weighted", "chance"]]
          .to_string(index=False, float_format=lambda x: f"{x:.4f}" if pd.notna(x) else "  n/a"))

[missing] results\text_cls_v2\sentiment_results.json
[missing] results\text_cls_v2\relation_results.json
[missing] results\text_cls_v2\corpus_results.json


KeyError: 'task'

## NR vs TSR Text Classification

Binary classification: can text features alone distinguish **Normal Reading (NR)** from **Task-Specific Reading (TSR)**?

| Label | ZuCo task key | Description |
|---|---|---|
| NR  | task1 | Movie reviews (sentiment analysis, normal reading) |
| NR  | task2 | Wikipedia biographies (normal reading) |
| TSR | task3 | Wikipedia biographies (task-specific reading, comprehension questions) |

**Source**: `zuco_label_8variants.df` (1.7 MB, text + labels only — no EEG needed)

**Three experiments**:
- All NR (task1+task2) vs TSR (task3) — 5-fold CV
- task1 vs task3 — different text genres (movie reviews vs biographies)
- task2 vs task3 — **same text genre**, different reading paradigm (hardest setting)

In [4]:
!python ../text_cls_baseline/run_nr_tsr.py \
    --label_df_path data/tmp/zuco_label_8variants.df \
    --output_dir    results/nr_tsr_cls \
    2>&1


  NR vs TSR Text Classification
  Chance baseline : 0.5000
  Data            : data/tmp/zuco_label_8variants.df
  n=1556  distribution={'NR': 989, 'TSR': 567}

[TF-IDF + LR]  5-fold CV  NR vs TSR  (all tasks)
    Fold 1/5  acc=0.7340  f1_macro=0.7248
    Fold 2/5  acc=0.7556  f1_macro=0.7465
    Fold 3/5  acc=0.7524  f1_macro=0.7428
    Fold 4/5  acc=0.7138  f1_macro=0.7062
    Fold 5/5  acc=0.7170  f1_macro=0.7057
  CV mean  acc=0.7346±0.0173  f1_macro=0.7252±0.0173
              precision    recall  f1-score   support

          NR       0.84      0.72      0.78       989
         TSR       0.61      0.75      0.67       567

    accuracy                           0.73      1556
   macro avg       0.72      0.74      0.73      1556
weighted avg       0.75      0.73      0.74      1556


[TF-IDF + LR]  5-fold CV  task1 (NR) vs task3 (TSR)
  n=967  dist={'NR': 400, 'TSR': 567}
    Fold 1/5  acc=0.9845  f1_macro=0.9841
    Fold 2/5  acc=0.9381  f1_macro=0.9359
    Fold 3/5  acc=0.9845 

In [5]:
"""Display NR vs TSR results."""
import json, pathlib

fpath = pathlib.Path("results/nr_tsr_cls/nr_tsr_results.json")
if not fpath.exists():
    print("[missing] Run the cell above first.")
else:
    r = json.loads(fpath.read_text())
    print(f"Task: NR vs TSR  |  n={r['n_total']}  dist={r['distribution']}")
    print(f"Chance baseline: {r['chance_baseline']:.4f}")
    print(f"\nNote: {r['note']}\n")

    for key, label in [
        ("full_nr_vs_tsr",        "All NR (task1+task2) vs TSR (task3)"),
        ("task1_nr_vs_task3_tsr", "task1-NR vs task3-TSR  [different genres]"),
        ("task2_nr_vs_task3_tsr", "task2-NR vs task3-TSR  [same genre, hard]"),
    ]:
        cv = r[key]
        print(f"[TF-IDF+LR] {label}")
        print(f"  CV  acc={cv['mean_acc']:.4f}±{cv['std_acc']:.4f}  "
              f"f1_macro={cv['mean_f1_macro']:.4f}±{cv['std_f1_macro']:.4f}")
        print(cv["pooled"]["report"])

Task: NR vs TSR  |  n=1556  dist={'NR': 989, 'TSR': 567}
Chance baseline: 0.5000

Note: NR includes task1 (movie reviews) and task2 (Wikipedia biographies). TSR is task3 (Wikipedia biographies, comprehension questions). task2 vs task3 is the hardest setting: same text type, different paradigm.

[TF-IDF+LR] All NR (task1+task2) vs TSR (task3)
  CV  acc=0.7346±0.0173  f1_macro=0.7252±0.0173
              precision    recall  f1-score   support

          NR       0.84      0.72      0.78       989
         TSR       0.61      0.75      0.67       567

    accuracy                           0.73      1556
   macro avg       0.72      0.74      0.73      1556
weighted avg       0.75      0.73      0.74      1556

[TF-IDF+LR] task1-NR vs task3-TSR  [different genres]
  CV  acc=0.9721±0.0208  f1_macro=0.9713±0.0215
              precision    recall  f1-score   support

          NR       0.96      0.97      0.97       400
         TSR       0.98      0.97      0.98       567

    accuracy   